<center><h1>Практическая работа №7</h1></center>

<center><h2>Тема работы: "Кластерный анализ и снижение размерности данных"</h2></center>

<h5>Цель работы: применить три алгоритма кластеризации (K-Means, Agglomerative Clustering, DBSCAN) на исходных данных и данных после снижения размерности (PCA), сравнить метрики качества и сделать выводы о наилучшей комбинации методов.</h5>

<h5>Ход работы:</h5>

<h4>1. Загрузка данных и первичный анализ.</h4>

In [53]:
import pandas as pd # библиотека для работы с табличными данными
import numpy as np # библиотека для численных вычислений

from sklearn.preprocessing import StandardScaler # инструмент для масштабирования данных (приведение к единому масштабу)
from sklearn.decomposition import PCA # метод понижения размерности (Principal Component Analysis)

from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN # алгоритмы кластеризации
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score # метрики для оценки качества кластеризации

from IPython.display import Markdown, display # для красивого отображения таблиц в Jupyter

In [54]:
path = r"A:\\Programs\\M_VS_Code_projects\\jupyter\\IPYNB-machine-learning\\cardekho.csv"
df = pd.read_csv(path) # загружает данные из CSV-файла в DataFrame pandas
df.info() # выводит сводную информацию о DataFrame: структуру, типы данных и количество ненулевых значений

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8128 entries, 0 to 8127
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   name                8128 non-null   object 
 1   year                8128 non-null   int64  
 2   selling_price       8128 non-null   int64  
 3   km_driven           8128 non-null   int64  
 4   fuel                8128 non-null   object 
 5   seller_type         8128 non-null   object 
 6   transmission        8128 non-null   object 
 7   owner               8128 non-null   object 
 8   mileage(km/ltr/kg)  7907 non-null   float64
 9   engine              7907 non-null   float64
 10  max_power           7913 non-null   object 
 11  seats               7907 non-null   float64
dtypes: float64(3), int64(3), object(6)
memory usage: 762.1+ KB


In [55]:
df.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage(km/ltr/kg),engine,max_power,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.40,1248.0,74,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14,1498.0,103.52,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.70,1497.0,78,5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.00,1396.0,90,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.10,1298.0,88.2,5.0


In [56]:
# Выбор числовых признаков для кластеризации
# Отбираем признаки, которые могут влиять на стоимость автомобиля
numeric_features = ['year', 'selling_price', 'km_driven', 'mileage(km/ltr/kg)', 
                    'engine', 'max_power', 'seats']

# Создаем копию датасета с числовыми признаками
df_numeric = df[numeric_features].copy()

# Преобразование строковых значений в числовые (если есть)
df_numeric['max_power'] = pd.to_numeric(df_numeric['max_power'], errors='coerce')
df_numeric['mileage(km/ltr/kg)'] = pd.to_numeric(df_numeric['mileage(km/ltr/kg)'], errors='coerce')

# Удаление строк с пропущенными значениями
df_clean = df_numeric.dropna()

print(f"Размер данных после очистки: {df_clean.shape}")
print(f"Количество удаленных строк: {len(df_numeric) - len(df_clean)}")

Размер данных после очистки: (7906, 7)
Количество удаленных строк: 222


<h4>2. Масштабирование данных.</h4>

In [57]:
scaler = StandardScaler() # создает объект для стандартизации данных (приведения к среднему 0 и стандартному отклонению 1)
X_scaled = scaler.fit_transform(df_clean) # сначала вычисляет параметры масштабирования (среднее и стандартное отклонение), затем применяет преобразование к данным

df_scaled = pd.DataFrame(X_scaled, columns=numeric_features)
print("Масштабированные данные (первые 5 строк):")
df_scaled.head()

Масштабированные данные (первые 5 строк):


,year,selling_price,km_driven,mileage(km/ltr/kg),engine,max_power,seats
0,0.004158,-0.245613,1.343777,0.986157,-0.418188,-0.492024,-0.434128
1,0.004158,-0.343950,0.894744,0.426198,0.077980,0.333827,-0.434128
2,-2.066530,-0.604542,1.246926,-0.426129,0.075995,-0.380120,-0.434128
3,-1.031186,-0.522185,1.018008,0.887050,-0.124457,-0.044408,-0.434128
4,-1.807694,-0.638960,0.894744,-0.822561,-0.318955,-0.094765,-0.434128


<h4>3. Кластеризация на исходных масштабированных данных.</h4>

<h4>3.1. K-Means кластеризация.</h4>

<h5>
K-Means (K-средних) — это один из самых простых и популярных алгоритмов кластеризации, который разделяет данные на заданное количество групп (кластеров) K.<br>
Как работает:
<ol>
    <li>Инициализация: Алгоритм случайным образом выбирает K точек в качестве центров кластеров</li>
    <li>Назначение: Каждая точка данных приписывается к ближайшему центру</li>
    <li>Обновление: Центры кластеров пересчитываются как среднее всех точек в кластере</li>
    <li>Повторение: Шаги 2-3 повторяются, пока центры кластеров не перестанут меняться или изменения не станут минимальными</li>
</ol>
</h5>

In [58]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
# n_clusters=3 — количество кластеров, на которое нужно разбить данные
# n_init=10 — количество запусков алгоритма с разными начальными центрами (выбирается лучший результат)
labels_kmeans = kmeans.fit_predict(X_scaled) # обучает модель и возвращает метки кластеров для каждой точки

# Метрики
sil_kmeans = silhouette_score(X_scaled, labels_kmeans) # вычисляет силуэтный коэффициент (чем ближе к 1, тем лучше)
db_kmeans = davies_bouldin_score(X_scaled, labels_kmeans) # вычисляет индекс Дэвиса-Болдина (чем меньше, тем лучше)
ch_kmeans = calinski_harabasz_score(X_scaled, labels_kmeans) # вычисляет индекс Калински-Харабаса (чем больше, тем лучше)

print("-------- K-Means (без PCA) --------")
print(f"Silhouette Score: {sil_kmeans:.4f}")
print(f"Davies-Bouldin Score: {db_kmeans:.4f}")
print(f"Calinski-Harabasz Score: {ch_kmeans:.2f}")
print("-" * 35)

-------- K-Means (без PCA) --------
Silhouette Score: 0.4484
Davies-Bouldin Score: 1.0106
Calinski-Harabasz Score: 3203.61
-----------------------------------


<h4>3.2. Agglomerative Clustering (Иерархическая кластеризация).</h4>

<h5>
Agglomerative Clustering (Иерархическая кластеризация) — это алгоритм кластеризации, который строит иерархию кластеров, объединяя объекты в группы на основе их близости.<br>
Как работает:
<ol>
    <li>Инициализация: Каждая точка данных изначально рассматривается как отдельный кластер</li>
    <li>Поиск ближайших кластеров: Алгоритм вычисляет расстояния между всеми парами кластеров и находит два самых близких</li>
    <li>Объединение: Два ближайших кластера объединяются в один новый кластер</li>
    <li>Обновление матрицы расстояний: Расстояния между новым кластером и остальными кластерами пересчитываются (с использованием выбранного метода linkage: ward, complete, average или single)</li>
    <li>Повторение: Шаги 2-4 повторяются, пока все точки не объединятся в один кластер или не будет достигнуто заданное количество кластеров</li>
</ol>
</h5>

In [59]:
agglo = AgglomerativeClustering(n_clusters=3, linkage='ward')
# ward объединяет те кластеры, после слияния которых данные внутри нового кластера останутся максимально похожими друг на друга.
labels_agglo = agglo.fit_predict(X_scaled)

# Метрики
sil_agglo = silhouette_score(X_scaled, labels_agglo)
db_agglo = davies_bouldin_score(X_scaled, labels_agglo)
ch_agglo = calinski_harabasz_score(X_scaled, labels_agglo)

print("-- Agglomerative Clustering (без PCA) --")
print(f"Silhouette Score: {sil_agglo:.4f}")
print(f"Davies-Bouldin Score: {db_agglo:.4f}")
print(f"Calinski-Harabasz Score: {ch_agglo:.2f}")
print("-" * 40)

-- Agglomerative Clustering (без PCA) --
Silhouette Score: 0.4641
Davies-Bouldin Score: 0.9172
Calinski-Harabasz Score: 2923.44
----------------------------------------


<h4>3.3. DBSCAN кластеризация.</h4>

<h5>
DBSCAN (Density-Based Spatial Clustering of Applications with Noise) — это алгоритм кластеризации, основанный на плотности распределения точек. Он не требует заранее задавать количество кластеров и может находить кластеры произвольной формы.<br>
Как работает:
<ol>
    <li>Определение соседей: Для каждой точки алгоритм проверяет, сколько других точек находится в её окрестности радиуса eps</li>
    <li>Поиск ядерных точек: Если в окрестности точки находится не менее min_samples точек, она становится ядром кластера</li>
    <li>Расширение кластеров: Все точки, достижимые из ядра (находящиеся на расстоянии eps друг от друга), включаются в тот же кластер</li>
    <li>Определение шума: Точки, которые не попадают ни в один кластер, помечаются как шум (метка -1)</li>
</ol>
</h5>

In [60]:
dbscan = DBSCAN(eps=0.7, min_samples=10)
# eps — максимальное расстояние между двумя точками, чтобы они считались соседями
# min_samples — минимальное количество точек в окрестности для образования кластера
labels_dbscan = dbscan.fit_predict(X_scaled)

# Количество кластеров (исключая шум, который помечен как -1)
n_clusters_dbscan = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0) # set(labels_dbscan) — создает множество уникальных меток кластеров
n_noise_dbscan = list(labels_dbscan).count(-1) # .count(-1) — подсчитывает количество точек, помеченных как шум (метка -1)

print("--------- DBSCAN (без PCA) --------")
print(f"Количество кластеров: {n_clusters_dbscan}")
print(f"Количество точек шума: {n_noise_dbscan}\n")

# Метрики рассчитываются только для точек, не являющихся шумом
if n_clusters_dbscan > 1:
    mask = labels_dbscan != -1 # создает булеву маску: True для точек, не являющихся шумом
    sil_dbscan = silhouette_score(X_scaled[mask], labels_dbscan[mask]) # X_scaled[mask] — выбирает только те строки, где маска равна True (отсеивает шум)
    db_dbscan = davies_bouldin_score(X_scaled[mask], labels_dbscan[mask])
    ch_dbscan = calinski_harabasz_score(X_scaled[mask], labels_dbscan[mask])

    print(f"Silhouette Score: {sil_dbscan:.4f}")
    print(f"Davies-Bouldin Score: {db_dbscan:.4f}")
    print(f"Calinski-Harabasz Score: {ch_dbscan:.2f}")
else:
    print("DBSCAN не смог найти кластеры (все точки - шум или один кластер).")

print("-" * 35)

--------- DBSCAN (без PCA) --------
Количество кластеров: 28
Количество точек шума: 755

Silhouette Score: -0.0317
Davies-Bouldin Score: 0.8953
Calinski-Harabasz Score: 409.54
-----------------------------------


<h4>4. Снижение размерности с помощью PCA.</h4>
<h5>PCA (Principal Component Analysis — Метод главных компонент) — это алгоритм уменьшения размерности данных, который позволяет "сжать" информацию из множества признаков в меньшее количество новых признаков (главных компонент) с минимальными потерями информации.</h5>

In [61]:
# Определяем количество компонент, объясняющих 85% дисперсии
pca_variance = PCA(random_state=42)
pca_variance.fit(X_scaled)

# Накопленная объясненная дисперсия
cumulative_variance = np.cumsum(pca_variance.explained_variance_ratio_) # np.cumsum() — накопленная сумма долей дисперсии. Увидеть, сколько информации сохраняется при добавлении компонент.
# explained_variance_ratio_ — доля дисперсии, объясняемая каждой отдельной компонентой. Понять "важность" каждой компоненты.

# Находим количество компонент для 85% дисперсии
n_components_85 = np.argmax(cumulative_variance >= 0.85) + 1 # np.argmax(... >= 0.85) + 1 — индекс, где накопленная дисперсия впервые ≥85%.

print("=== Анализ объясненной дисперсии PCA ===")
print(f"Количество компонент для 85% дисперсии: {n_components_85}")
print(f"Объясненная дисперсия по компонентам:")
for i, var in enumerate(pca_variance.explained_variance_ratio_[:5], 1):
    print(f"  PC{i}: {var:.2%}")

# Применяем PCA с выбранным количеством компонент
pca = PCA(n_components=n_components_85, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"\nФорма данных после PCA: {X_pca.shape}")
print(f"Общая объясненная дисперсия: {cumulative_variance[n_components_85-1]:.2%}")
# Дисперсия — это мера того, насколько сильно разбросаны данные. Если данные "кучкуются" в одной точке — дисперсия маленькая. Если данные "размазаны" широко — дисперсия большая.
# Накопленная объяснённая дисперсия — суммарная доля дисперсии первых k компонент. Критерий качества снижения размерности.
# Объяснённая дисперсия — мера сохранённой информации после проекции на главные компоненты. Оценка того, насколько хорошо PCA аппроксимирует исходные данные.

=== Анализ объясненной дисперсии PCA ===
Количество компонент для 85% дисперсии: 4
Объясненная дисперсия по компонентам:
  PC1: 40.04%
  PC2: 28.15%
  PC3: 11.88%
  PC4: 10.29%
  PC5: 4.31%

Форма данных после PCA: (7906, 4)
Общая объясненная дисперсия: 90.37%


<h4>5. Кластеризация на данных после PCA.</h4>

<h5>5.1. K-Means на PCA-данных.</h5>

In [62]:
kmeans_pca = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_kmeans_pca = kmeans_pca.fit_predict(X_pca)

sil_kmeans_pca = silhouette_score(X_pca, labels_kmeans_pca)
db_kmeans_pca = davies_bouldin_score(X_pca, labels_kmeans_pca)
ch_kmeans_pca = calinski_harabasz_score(X_pca, labels_kmeans_pca)

print("--------- K-Means (с PCA) ---------")
print(f"Silhouette Score: {sil_kmeans_pca:.4f}")
print(f"Davies-Bouldin Score: {db_kmeans_pca:.4f}")
print(f"Calinski-Harabasz Score: {ch_kmeans_pca:.2f}")
print("-" * 35)

--------- K-Means (с PCA) ---------
Silhouette Score: 0.4798
Davies-Bouldin Score: 0.8905
Calinski-Harabasz Score: 3835.90
-----------------------------------


<h5>5.2. Agglomerative Clustering на PCA-данных.</h5>

In [63]:
agglo_pca = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels_agglo_pca = agglo_pca.fit_predict(X_pca)

sil_agglo_pca = silhouette_score(X_pca, labels_agglo_pca)
db_agglo_pca = davies_bouldin_score(X_pca, labels_agglo_pca)
ch_agglo_pca = calinski_harabasz_score(X_pca, labels_agglo_pca)

print("--- Agglomerative Clustering (с PCA) ---")
print(f"Silhouette Score: {sil_agglo_pca:.4f}")
print(f"Davies-Bouldin Score: {db_agglo_pca:.4f}")
print(f"Calinski-Harabasz Score: {ch_agglo_pca:.2f}")
print("-" * 40)

--- Agglomerative Clustering (с PCA) ---
Silhouette Score: 0.4774
Davies-Bouldin Score: 0.8378
Calinski-Harabasz Score: 3447.86
----------------------------------------


<h5>5.3. DBSCAN на PCA-данных.</h5>

In [64]:
dbscan_pca = DBSCAN(eps=0.5, min_samples=10) # eps изменен на 0.5 (было 0.7), так как после PCA данные имеют другую структуру и масштаб.
labels_dbscan_pca = dbscan_pca.fit_predict(X_pca)

n_clusters_dbscan_pca = len(set(labels_dbscan_pca)) - (1 if -1 in labels_dbscan_pca else 0)
n_noise_dbscan_pca = list(labels_dbscan_pca).count(-1)

print("---------- DBSCAN (с PCA) ---------")
print(f"Количество кластеров: {n_clusters_dbscan_pca}")
print(f"Количество точек шума: {n_noise_dbscan_pca}\n")

if n_clusters_dbscan_pca > 1:
    mask_pca = labels_dbscan_pca != -1
    sil_dbscan_pca = silhouette_score(X_pca[mask_pca], labels_dbscan_pca[mask_pca])
    db_dbscan_pca = davies_bouldin_score(X_pca[mask_pca], labels_dbscan_pca[mask_pca])
    ch_dbscan_pca = calinski_harabasz_score(X_pca[mask_pca], labels_dbscan_pca[mask_pca])

    print(f"Silhouette Score: {sil_dbscan_pca:.4f}")
    print(f"Davies-Bouldin Score: {db_dbscan_pca:.4f}")
    print(f"Calinski-Harabasz Score: {ch_dbscan_pca:.2f}")
else:
    print("DBSCAN (PCA) не смог найти кластеры.")

print("-" * 35)

---------- DBSCAN (с PCA) ---------
Количество кластеров: 19
Количество точек шума: 712

Silhouette Score: 0.0212
Davies-Bouldin Score: 0.6960
Calinski-Harabasz Score: 549.62
-----------------------------------


<h4>6. Сравнение результатов и выводы.</h4>

<h5>
Silhouette Score (Коэффициент силуэта) — это метрика, используемая для оценки качества кластеризации. Она показывает, насколько хорошо объекты разделены на кластеры и насколько каждый объект похож на свой собственный кластер по сравнению с другими кластерами. Чем ближе к 1, тем лучше.</br>

Davies-Bouldin Index (Индекс Дэвиса-Болдина) — это метрика для оценки качества кластеризации, которая измеряет "похожесть" между кластерами. Она показывает, насколько кластеры компактны внутри себя и насколько далеко они расположены друг от друга. Чем ниже значение, тем лучше разделение кластеров.

Calinski-Harabasz Index (Индекс Калински-Харабаса) — это метрика для оценки качества кластеризации, основанная на анализе дисперсии. Она оценивает, насколько хорошо кластеры разделены и насколько они плотные. Чем выше значение, тем лучше.

В итоге:
<ul>
    <li>Silhouette ↑ — объекты похожи на свои кластеры</li>
    <li>Davies-Bouldin ↓ — кластеры минимально пересекаются</li>
    <li>Calinski-Harabasz ↑ — кластеры очень плотные и далеко отстоят друг от друга</li>
</ul>
</h5>

In [65]:
# Создание сводной таблицы результатов для наглядного сравнения
table_md = """
| Модель | Silhouette ↑ | Davies-Bouldin ↓ | Calinski-Harabasz ↑ |
|--------|--------------|------------------|---------------------|
| **K-Means** | {:.4f} | {:.4f} | {:.2f} |
| **K-Means (PCA)** | **{:.4f}** | **{:.4f}** | **{:.2f}** |
| **Agglomerative** | {:.4f} | {:.4f} | {:.2f} |
| **Agglomerative (PCA)** | {:.4f} | {:.4f} | {:.2f} |
| **DBSCAN** | {:.4f} | {:.4f} | {:.2f} |
| **DBSCAN (PCA)** | {:.4f} | {:.4f} | {:.2f} |
""".format(
    sil_kmeans, db_kmeans, ch_kmeans,
    sil_kmeans_pca, db_kmeans_pca, ch_kmeans_pca,
    sil_agglo, db_agglo, ch_agglo,
    sil_agglo_pca, db_agglo_pca, ch_agglo_pca,
    sil_dbscan, db_dbscan, ch_dbscan,
    sil_dbscan_pca, db_dbscan_pca, ch_dbscan_pca
)

display(Markdown(table_md))

# Анализ влияния PCA на качество кластеризации
print("------------------------- Анализ влияния PCA -------------------------")
print("1. K-Means:")
print(f"   Silhouette: {sil_kmeans:.4f} → {sil_kmeans_pca:.4f} ({'↑ улучшилось' if sil_kmeans_pca > sil_kmeans else '↓ ухудшилось'})")
print(f"   Davies-Bouldin: {db_kmeans:.4f} → {db_kmeans_pca:.4f} ({'↓ улучшилось' if db_kmeans_pca < db_kmeans else '↑ ухудшилось'})")
print(f"   Calinski-Harabasz: {ch_kmeans:.2f} → {ch_kmeans_pca:.2f} ({'↑ улучшилось' if ch_kmeans_pca > ch_kmeans else '↓ ухудшилось'})")

print("\n2. Agglomerative Clustering:")
print(f"   Silhouette: {sil_agglo:.4f} → {sil_agglo_pca:.4f} ({'↑ улучшилось' if sil_agglo_pca > sil_agglo else '↓ ухудшилось'})")
print(f"   Davies-Bouldin: {db_agglo:.4f} → {db_agglo_pca:.4f} ({'↓ улучшилось' if db_agglo_pca < db_agglo else '↑ ухудшилось'})")
print(f"   Calinski-Harabasz: {ch_agglo:.2f} → {ch_agglo_pca:.2f} ({'↑ улучшилось' if ch_agglo_pca > ch_agglo else '↓ ухудшилось'})")

print("\n3. DBSCAN:")
print(f"   Silhouette: {sil_dbscan:.4f} → {sil_dbscan_pca:.4f} ({'↑ улучшилось' if sil_dbscan_pca > sil_dbscan else '↓ ухудшилось'})")
print(f"   Davies-Bouldin: {db_dbscan:.4f} → {db_dbscan_pca:.4f} ({'↓ улучшилось' if db_dbscan_pca < db_dbscan else '↑ ухудшилось'})")
print(f"   Calinski-Harabasz: {ch_dbscan:.2f} → {ch_dbscan_pca:.2f} ({'↑ улучшилось' if ch_dbscan_pca > ch_dbscan else '↓ ухудшилось'})")


| Модель | Silhouette ↑ | Davies-Bouldin ↓ | Calinski-Harabasz ↑ |
|--------|--------------|------------------|---------------------|
| **K-Means** | 0.4484 | 1.0106 | 3203.61 |
| **K-Means (PCA)** | **0.4798** | **0.8905** | **3835.90** |
| **Agglomerative** | 0.4641 | 0.9172 | 2923.44 |
| **Agglomerative (PCA)** | 0.4774 | 0.8378 | 3447.86 |
| **DBSCAN** | -0.0317 | 0.8953 | 409.54 |
| **DBSCAN (PCA)** | 0.0212 | 0.6960 | 549.62 |


------------------------- Анализ влияния PCA -------------------------
1. K-Means:
   Silhouette: 0.4484 → 0.4798 (↑ улучшилось)
   Davies-Bouldin: 1.0106 → 0.8905 (↓ улучшилось)
   Calinski-Harabasz: 3203.61 → 3835.90 (↑ улучшилось)

2. Agglomerative Clustering:
   Silhouette: 0.4641 → 0.4774 (↑ улучшилось)
   Davies-Bouldin: 0.9172 → 0.8378 (↓ улучшилось)
   Calinski-Harabasz: 2923.44 → 3447.86 (↑ улучшилось)

3. DBSCAN:
   Silhouette: -0.0317 → 0.0212 (↑ улучшилось)
   Davies-Bouldin: 0.8953 → 0.6960 (↓ улучшилось)
   Calinski-Harabasz: 409.54 → 549.62 (↑ улучшилось)


<h5>
PCA положительно повлияло на качество кластеризации по следующим причинам:
<ul>
    <li>Устранение шума: PCA отфильтровывает признаки с низкой объясняющей способностью.</li>
    <li>Улучшение разделимости: В пространстве меньшей размерности кластеры становятся более компактными.</li>
    <li>Вычислительная эффективность: Меньше признаков = быстрее работа алгоритмов.</li>
</ul>
Лучшая комбинация: K-Means + PCA. Обоснование:
<ul>
    <li>Высокий Silhouette Score (>0.45) — кластеры хорошо разделены.</li>
    <li>Средний Davies-Bouldin — среднее перекрытие между кластерами.</li>
    <li>Высокий Calinski-Harabasz — плотные и компактные кластеры.</li>
</ul>
</h5>

<h5>Вывод: в ходе выполнения работы были применены три алгоритма кластеризации (K-Means, Agglomerative Clustering, DBSCAN) на исходных данных и данных после снижения размерности (PCA), были сравнение метрик качества и сделаны выводы о наилучшей комбинации методов.</h5>